# 02 · Datos públicos: descarga, remapeo, fusión y recortes

Este notebook **no tiene lógica propia**: cada celda corre un script de `scripts/datos_*.py`
(los mismos que se pueden lanzar desde la terminal) y muestra su salida en vivo.

| Paso | Script | Salida (en `DIR_DATOS`) |
|---|---|---|
| 1 | `datos_descargar.py` | `raw/<alias>/` (Roboflow Universe, YOLOv8) |
| 2 | `datos_remapeo.py` | `config/remapeo_clases.yaml` (borrador) |
| 3 | `datos_indexar.py` | `fusion/_indice_imagenes.csv` (pHash, grupos, splits) |
| 3b | `datos_color.py` | `fusion/_control_color.csv` (cajas con color incoherente) |
| 4 | `datos_fusionar.py` | `fusion/detector_1clase/`, `fusion/multiclase/` |
| 5 | `datos_recortes.py` | `recortes/<split>/<clase>/` + `indice_recortes.csv` |
| 6 | `datos_reporte.py` | `docs/datos.md` + grillas en `docs/datos/` |

Todos los pasos son idempotentes: la descarga no se repite y el pHash se cachea.
`DIR_DATOS` sale de `src/config.py` (variable `INVENTARIO_DATOS` si está definida).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src.config import DIR_DATOS  # noqa: E402

print("Raíz del repo:", RAIZ)
print("DIR_DATOS:    ", DIR_DATOS)


def correr(script: str, *args: str) -> None:
    """Corre scripts/<script> en un subproceso y muestra la salida en vivo."""
    entorno = {**os.environ, "PYTHONIOENCODING": "utf-8", "INVENTARIO_DATOS": str(DIR_DATOS)}
    cmd = [sys.executable, str(RAIZ / "scripts" / script), *args]
    with subprocess.Popen(cmd, cwd=RAIZ, env=entorno, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as p:
        for linea in p.stdout:
            if "Downloading Dataset" in linea or "Extracting Dataset" in linea:
                continue  # barras de progreso del paquete roboflow
            print(linea, end="")
    if p.returncode:
        raise RuntimeError(f"{script} terminó con código {p.returncode}")

## 1. Descarga

Primero se consultan los metadatos (licencia, versiones, clases) sin descargar nada.
La clave `ROBOFLOW_API_KEY` se lee del `.env` (nunca se imprime).

In [ ]:
correr("datos_descargar.py", "--inspeccionar")

In [ ]:
correr("datos_descargar.py")  # salta lo ya descargado; --forzar para repetir

## 2. Remapeo de clases (borrador)

Agrega al YAML las clases nuevas con una sugerencia automática y respeta lo ya revisado.
Destinos: una de las 8 clases, `solo_detector` o `descartar`.

In [ ]:
correr("datos_remapeo.py")

## 3. Índice, grupos anti-fuga y splits

Grupo = misma foto original (sin el sufijo `.rf.<hash>` de Roboflow) o pHash idéntico.
El split se asigna por grupo y se verifica que no haya fugas.

In [ ]:
correr("datos_indexar.py")

## 3 b. Control de color por clase

Descarta del multiclase y de los recortes las cajas cuyo color no coincide con la clase
(etiquetas equivocadas, variantes de otro color, botellas vacías, copias en grises).
Guarda `fusion/_control_color.csv` y las grillas `docs/datos/color_*.jpg`.

In [ ]:
correr("datos_color.py")

## 4. Datasets YOLO fusionados

In [ ]:
correr("datos_fusionar.py")

## 5. Recortes para el clasificador

In [ ]:
correr("datos_recortes.py", "--margen", "0.05", "--min-lado", "32", "--min-nitidez", "40")

## 6. Reporte de calidad y conteos

In [ ]:
correr("datos_reporte.py")

In [ ]:
from IPython.display import Markdown, display

display(Markdown((RAIZ / "docs" / "datos.md").read_text(encoding="utf-8").replace("](datos/", "](../docs/datos/")))

### Grillas de muestras (verde = nuestra clase, amarillo = solo_detector, rojo = descartar)

In [ ]:
from IPython.display import Image as ImagenIPy

for ruta in sorted((RAIZ / "docs" / "datos").glob("*.jpg")):
    print(ruta.name)
    display(ImagenIPy(filename=str(ruta), width=900))

## 7. Tests del pipeline (mini dataset sintético)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_datos.py", "-q"], cwd=RAIZ,
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-3000:], r.stderr[-2000:])